# ✈️ US Flight Delay Predictor — Full Dataset
### Big Data Analysis — AIE College

| Item | Detail |
|------|--------|
| Dataset | US Flight Delays — Full Range (all years, all CSV files) |
| Engine | Apache Spark via PySpark |
| Models | Linear Regression · Decision Tree · Random Forest · GBT |
| Metrics | RMSE · MAE · R² |

**Goal:** Load ALL CSV files from the dataset folder into one unified Spark DataFrame, clean and analyse millions of flight records, and compare 4 ML models to find the best predictor of arrival delay.

---

## ⚙️ Section 1 — Setup & SparkSession

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, avg, count, when, isnan, isnull,
    round as spark_round, abs as spark_abs,
    corr, stddev,
    min as spark_min, max as spark_max,
    percentile_approx, lit, year, month,
    dayofweek, to_date, concat_ws, input_file_name
)
from pyspark.sql.types import DoubleType, IntegerType, StringType

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import pandas as pd
import numpy as np
import os, glob, time, warnings
warnings.filterwarnings('ignore')

# ── Consistent plot style ─────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#F8F9FA',
    'axes.grid':        True,
    'grid.alpha':       0.4,
    'font.size':        11,
    'axes.titlesize':   13,
    'axes.titleweight': 'bold',
})
PAL = ['#1565C0','#C62828','#2E7D32','#E65100','#6A1B9A','#00695C','#4E342E']
print('✅ Imports OK')

✅ Imports OK


In [2]:
spark = (
    SparkSession.builder
    .appName('US Flight Delay Predictor — Full Range')
    .config('spark.driver.memory',            '6g')
    .config('spark.executor.memory',          '4g')
    .config('spark.sql.shuffle.partitions',   '100')
    .config('spark.sql.adaptive.enabled',     'true')
    .config('spark.driver.maxResultSize',     '2g')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('ERROR')
print(f'Spark {spark.version} ready')
spark

Spark 4.1.1 ready


---
## 📂 Section 2 — Load ALL CSV Files

### ⚠️ IMPORTANT — set your folder path below

Put the path to the folder that contains **all your downloaded CSV files**.

Examples:
```
DATA_FOLDER = 'data'                          # subfolder next to this notebook
DATA_FOLDER = r'C:\Users\you\Downloads\flights'  # absolute Windows path
DATA_FOLDER = '/home/you/datasets/flights'    # absolute Linux/Mac path
```
Spark will automatically read **every `.csv` file** inside that folder in one call — no loops needed.

In [3]:
# ── SET THIS TO YOUR FOLDER ───────────────────────────────
DATA_FOLDER = r"C:\Users\outis\Desktop\College\Lvl 300\2nd term\Big Data\Project\Us flight delay"     # folder containing all CSV files
# ─────────────────────────────────────────────────────────

# Discover all CSV files in the folder
csv_files = glob.glob(os.path.join(DATA_FOLDER, '*.csv'))
print(f'Found {len(csv_files)} CSV file(s):')
for f in sorted(csv_files):
    size_mb = os.path.getsize(f) / (1024**2)
    print(f'  {os.path.basename(f):<45}  {size_mb:>7.1f} MB')

total_mb = sum(os.path.getsize(f) for f in csv_files) / (1024**2)
print(f'\nTotal raw data : {total_mb:.1f} MB  ({total_mb/1024:.2f} GB)')

Found 6 CSV file(s):
  Airlines.csv                                       0.0 MB
  Combined_Flights_2018.csv                       1900.0 MB
  Combined_Flights_2019.csv                       2690.8 MB
  Combined_Flights_2020.csv                       1665.7 MB
  Combined_Flights_2021.csv                       2111.7 MB
  Combined_Flights_2022.csv                       1349.7 MB

Total raw data : 9718.0 MB  (9.49 GB)


In [4]:
# ── Read ALL CSVs at once with wildcard ───────────────────
# Spark merges all files into one DataFrame automatically.
# mergeSchema=True handles slight column differences across years.

t0 = time.time()
df_raw = spark.read.csv(
    os.path.join(DATA_FOLDER, '*.csv'),
    header=True,
    inferSchema=True,
    mergeSchema=True,
)
print(f'Read time : {time.time()-t0:.1f}s')
print(f'Raw rows  : {df_raw.count():,}')
print(f'Columns   : {len(df_raw.columns)}')
df_raw.printSchema()

TypeError: DataFrameReader.csv() got an unexpected keyword argument 'mergeSchema'

In [ ]:
# Preview raw data
df_raw.show(5, truncate=18)

---
## 🧹 Section 3 — Data Cleaning & Preprocessing

**8-step pipeline:**
1. Standardise column names (handles all Kaggle naming variations)
2. Select & cast relevant columns
3. Remove cancelled / diverted flights
4. Remove rows where the label (`arr_delay`) is null
5. Report & handle missing values
6. Remove outliers via IQR (×3)
7. Engineer time features (year, month, day-of-week)
8. Cache the clean DataFrame

In [ ]:
# ── Step 1: Standardise column names ──────────────────────
df = df_raw
rename_map = {c: c.strip().lower().replace(' ', '_').replace('/', '_')
              for c in df.columns}
for old, new in rename_map.items():
    if old != new:
        df = df.withColumnRenamed(old, new)

# Alias map — covers naming differences across BTS / Kaggle versions
ALIASES = {
    'arr_delay':    ['arr_delay','arrdelay','arrival_delay','arr_delay_new'],
    'dep_delay':    ['dep_delay','depdelay','departure_delay','dep_delay_new'],
    'distance':     ['distance','dist'],
    'airline':      ['airline','mkt_carrier','carrier','op_carrier',
                     'reporting_airline','unique_carrier','iata_code_operating_airline'],
    'origin':       ['origin','origin_airport'],
    'dest':         ['dest','destination','dest_airport'],
    'cancelled':    ['cancelled','cancellation'],
    'diverted':     ['diverted'],
    'air_time':     ['air_time','airtime','actual_elapsed_time'],
    'taxi_out':     ['taxi_out','taxiout'],
    'crs_dep_time': ['crs_dep_time','scheduled_departure','crs_dep_time'],
    'fl_date':      ['fl_date','flightdate','flight_date','date'],
    'year':         ['year'],
    'month':        ['month'],
    'day_of_week':  ['day_of_week','dayofweek','day_of_week'],
}

existing = set(df.columns)
for standard, candidates in ALIASES.items():
    if standard not in existing:                # only if not already present
        for c in candidates:
            if c in existing:
                df = df.withColumnRenamed(c, standard)
                break

have = [s for s in ALIASES if s in df.columns]
print(f'Matched columns ({len(have)}): {have}')

In [ ]:
# ── Step 2: Select & cast columns ─────────────────────────
MUST_HAVE    = ['arr_delay', 'dep_delay', 'distance']
NICE_TO_HAVE = ['air_time', 'taxi_out', 'crs_dep_time']
CATEGORICAL  = ['airline', 'origin', 'dest']
TIME_COLS    = ['year', 'month', 'day_of_week', 'fl_date']
FILTER_COLS  = ['cancelled', 'diverted']

all_want = MUST_HAVE + NICE_TO_HAVE + CATEGORICAL + TIME_COLS + FILTER_COLS
select   = [c for c in all_want if c in df.columns]
df       = df.select(select)

# Cast numeric
for c in MUST_HAVE + NICE_TO_HAVE:
    if c in df.columns:
        df = df.withColumn(c, col(c).cast(DoubleType()))

for c in ['year', 'month', 'day_of_week']:
    if c in df.columns:
        df = df.withColumn(c, col(c).cast(IntegerType()))

print(f'Working with {len(df.columns)} columns: {df.columns}')

In [ ]:
# ── Step 3: Remove cancelled & diverted flights ────────────
before = df.count()

if 'cancelled' in df.columns:
    df = df.filter((col('cancelled') == 0) | col('cancelled').isNull())
    df = df.drop('cancelled')
if 'diverted' in df.columns:
    df = df.filter((col('diverted') == 0) | col('diverted').isNull())
    df = df.drop('diverted')

# ── Step 4: Drop rows where label is null ─────────────────
df = df.filter(col('arr_delay').isNotNull())

after = df.count()
print(f'Rows before : {before:,}')
print(f'Rows after  : {after:,}')
print(f'Removed     : {before - after:,}  ({(before-after)/before*100:.1f}%)')

In [ ]:
# ── Step 5: Missing value report & handling ────────────────
print('=== Missing Value Report ===')
total = df.count()
for c in df.columns:
    n   = df.filter(col(c).isNull() | (col(c).cast('string') == 'NA')
                    | (when(col(c).cast('double').isNotNull(),
                            isnan(col(c).cast('double'))).isNull() == False)
                   ).count()
    pct = n / total * 100
    bar = '█' * int(pct / 2)
    print(f'  {c:<20} {n:>9,}  {pct:5.1f}%  {bar}')

print()
# Drop rows missing critical numeric features
df = df.dropna(subset=['arr_delay', 'dep_delay', 'distance'])

# Fill optional numeric cols with median
optional_present = [c for c in NICE_TO_HAVE if c in df.columns]
for c in optional_present:
    try:
        med = df.approxQuantile(c, [0.5], 0.01)[0]
        df  = df.fillna({c: med})
    except:
        pass

# Fill categorical with 'Unknown'
for c in CATEGORICAL:
    if c in df.columns:
        df = df.fillna({c: 'Unknown'})

# Fill time cols with 0 if missing
for c in ['year','month','day_of_week']:
    if c in df.columns:
        df = df.fillna({c: 0})

print(f'Rows after null handling: {df.count():,}')

In [ ]:
# ── Step 6: Outlier removal (IQR × 3) ─────────────────────
def iqr_filter(df, c, factor=3.0):
    q1, q3 = df.approxQuantile(c, [0.25, 0.75], 0.005)
    iqr     = q3 - q1
    lo, hi  = q1 - factor*iqr, q3 + factor*iqr
    return df.filter((col(c) >= lo) & (col(c) <= hi)), lo, hi

before_out = df.count()
df, a_lo, a_hi = iqr_filter(df, 'arr_delay')
df, d_lo, d_hi = iqr_filter(df, 'dep_delay')
df, _, _        = iqr_filter(df, 'distance')

after_out = df.count()
print(f'arr_delay  kept in [{a_lo:.0f}, {a_hi:.0f}] min')
print(f'dep_delay  kept in [{d_lo:.0f}, {d_hi:.0f}] min')
print(f'Rows before outlier removal : {before_out:,}')
print(f'Rows after  outlier removal : {after_out:,}')
print(f'Outliers removed            : {before_out - after_out:,}')

In [ ]:
# ── Step 7: Derive time features if not already present ───
# Extract year / month / day_of_week from fl_date if needed
if 'fl_date' in df.columns:
    df = df.withColumn('fl_date_parsed', to_date(col('fl_date')))
    if 'year' not in df.columns or df.filter(col('year') == 0).count() > 1000:
        df = df.withColumn('year',        year('fl_date_parsed'))
    if 'month' not in df.columns or df.filter(col('month') == 0).count() > 1000:
        df = df.withColumn('month',       month('fl_date_parsed'))
    if 'day_of_week' not in df.columns:
        df = df.withColumn('day_of_week', dayofweek('fl_date_parsed'))
    df = df.drop('fl_date', 'fl_date_parsed')

# Hour of departure from crs_dep_time (format: HHMM → HH)
if 'crs_dep_time' in df.columns:
    df = df.withColumn(
        'dep_hour',
        (col('crs_dep_time') / 100).cast(IntegerType())
    ).drop('crs_dep_time')

# ── Step 8: Cache ──────────────────────────────────────────
df.cache()
TOTAL_ROWS = df.count()
print(f'✅ Clean DataFrame cached')
print(f'   Total rows    : {TOTAL_ROWS:,}')
print(f'   Final columns : {df.columns}')
df.show(5, truncate=15)

In [ ]:
# ── Year range summary ─────────────────────────────────────
if 'year' in df.columns:
    year_counts = (
        df.groupBy('year')
        .agg(count('*').alias('flights'))
        .orderBy('year')
        .toPandas()
    )
    print('=== Flights per Year in Clean Dataset ===')
    print(year_counts.to_string(index=False))
    print(f'\nYear range : {int(year_counts.year.min())} → {int(year_counts.year.max())}')
    print(f'Total      : {year_counts.flights.sum():,}')

---
## 📊 Section 4 — Exploratory Data Analysis
**10 visualizations** covering distributions, trends over time, airline performance, routes, and feature correlations.

In [ ]:
# ── Helper: safe sample to pandas ─────────────────────────
def to_pd(spark_df, n=50000, seed=42):
    frac = min(1.0, n / TOTAL_ROWS)
    return spark_df.sample(fraction=frac, seed=seed).toPandas().dropna()

print('Helper ready')

In [ ]:
# ── VIZ 1: Delay distributions ────────────────────────────
samp = to_pd(df.select('arr_delay','dep_delay'))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Flight Delay Distributions — Full Dataset', fontsize=15, fontweight='bold')

for ax, col_name, color, title in zip(
    axes,
    ['arr_delay', 'dep_delay'],
    [PAL[0], PAL[1]],
    ['Arrival Delay', 'Departure Delay']
):
    ax.hist(samp[col_name], bins=80, color=color, edgecolor='white', linewidth=0.4, alpha=0.85)
    ax.axvline(0,  color='black', lw=1.5, linestyle='--', label='On time')
    ax.axvline(samp[col_name].mean(), color='gold', lw=1.5, linestyle='-.',
               label=f'Mean: {samp[col_name].mean():.1f} min')
    ax.set_xlabel('Delay (minutes)')
    ax.set_ylabel('Flights')
    ax.set_title(title)
    ax.legend()
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x/1000:.0f}k'))

plt.tight_layout()
plt.savefig('viz_01_delay_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── VIZ 2: Flight status breakdown ────────────────────────
status_df = (
    df.withColumn('status',
        when(col('arr_delay') < -5,  'Early (>5 min)')
        .when(col('arr_delay') <= 15, 'On Time (±15 min)')
        .when(col('arr_delay') <= 60, 'Delayed (15–60 min)')
        .otherwise('Severely Delayed (>60 min)'))
    .groupBy('status').agg(count('*').alias('flights'))
    .toPandas()
)
order  = ['Early (>5 min)','On Time (±15 min)','Delayed (15–60 min)','Severely Delayed (>60 min)']
status_df = status_df.set_index('status').reindex(order).reset_index().dropna()
status_df['pct'] = (status_df['flights'] / status_df['flights'].sum() * 100).round(1)
status_colors = [PAL[2], PAL[0], PAL[3], PAL[1]]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Flight Status Breakdown — All Years', fontsize=15, fontweight='bold')

wedges, _, autotexts = axes[0].pie(
    status_df['flights'],
    labels=None,
    colors=status_colors[:len(status_df)],
    autopct='%1.1f%%',
    startangle=90, pctdistance=0.75,
    wedgeprops={'edgecolor':'white','linewidth':2}
)
axes[0].legend(wedges, status_df['status'], loc='lower center',
               bbox_to_anchor=(0.5, -0.1), fontsize=9)
axes[0].set_title('Proportion')

bars = axes[1].barh(status_df['status'], status_df['flights'],
                    color=status_colors[:len(status_df)], edgecolor='white')
for bar, row in zip(bars, status_df.itertuples()):
    axes[1].text(bar.get_width()*1.01, bar.get_y()+bar.get_height()/2,
                 f'{row.pct}%  ({row.flights/1e6:.1f}M)', va='center', fontsize=9)
axes[1].set_xlabel('Number of Flights')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x/1e6:.0f}M'))
axes[1].set_title('Count')

plt.tight_layout()
plt.savefig('viz_02_flight_status.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── VIZ 3: Average delay by year (trend over time) ────────
if 'year' in df.columns:
    yearly = (
        df.groupBy('year')
        .agg(
            spark_round(avg('arr_delay'), 2).alias('avg_arr'),
            spark_round(avg('dep_delay'), 2).alias('avg_dep'),
            (count(when(col('arr_delay') > 15, 1)) * 100.0
             / count('*')).alias('pct_delayed')
        )
        .orderBy('year')
        .toPandas()
        .dropna()
    )

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    fig.suptitle('Delay Trends Over the Years', fontsize=15, fontweight='bold')

    axes[0].plot(yearly['year'], yearly['avg_arr'], 'o-', color=PAL[0],
                 lw=2.5, markersize=7, label='Arrival')
    axes[0].plot(yearly['year'], yearly['avg_dep'], 's--', color=PAL[1],
                 lw=2, markersize=7, label='Departure')
    axes[0].axhline(0, color='grey', lw=0.8, linestyle=':')
    axes[0].fill_between(yearly['year'], yearly['avg_arr'], 0,
                         alpha=0.1, color=PAL[0])
    axes[0].set_xlabel('Year')
    axes[0].set_ylabel('Avg Delay (minutes)')
    axes[0].set_title('Average Delay per Year')
    axes[0].legend()
    axes[0].xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

    axes[1].bar(yearly['year'], yearly['pct_delayed'], color=PAL[3],
                edgecolor='white', width=0.6)
    axes[1].set_xlabel('Year')
    axes[1].set_ylabel('% of Flights Delayed (>15 min)')
    axes[1].set_title('Delay Rate per Year')
    axes[1].xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

    plt.tight_layout()
    plt.savefig('viz_03_yearly_trends.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(yearly.to_string(index=False))

In [ ]:
# ── VIZ 4: Monthly seasonality ────────────────────────────
if 'month' in df.columns:
    monthly = (
        df.groupBy('month')
        .agg(spark_round(avg('arr_delay'), 2).alias('avg_arr'),
             count('*').alias('flights'))
        .orderBy('month')
        .toPandas().dropna()
    )
    month_names = ['Jan','Feb','Mar','Apr','May','Jun',
                   'Jul','Aug','Sep','Oct','Nov','Dec']
    monthly['month_name'] = monthly['month'].apply(
        lambda m: month_names[int(m)-1] if 1 <= int(m) <= 12 else str(m))

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    fig.suptitle('Monthly Seasonality', fontsize=15, fontweight='bold')

    c_month = [PAL[1] if v == monthly['avg_arr'].max()
                else (PAL[2] if v == monthly['avg_arr'].min() else PAL[0])
                for v in monthly['avg_arr']]
    axes[0].bar(monthly['month_name'], monthly['avg_arr'],
                color=c_month, edgecolor='white')
    axes[0].axhline(monthly['avg_arr'].mean(), color='grey',
                    lw=1.2, linestyle='--', label='Annual avg')
    axes[0].set_ylabel('Avg Arrival Delay (min)')
    axes[0].set_title('Avg Delay by Month (red=worst, green=best)')
    axes[0].legend()

    axes[1].plot(monthly['month_name'], monthly['flights']/1e6,
                 'o-', color=PAL[4], lw=2.5, markersize=8)
    axes[1].fill_between(range(len(monthly)), monthly['flights']/1e6,
                         alpha=0.15, color=PAL[4])
    axes[1].set_ylabel('Flights (millions)')
    axes[1].set_title('Flight Volume by Month')

    plt.tight_layout()
    plt.savefig('viz_04_monthly_seasonality.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ── VIZ 5: Airline performance ────────────────────────────
if 'airline' in df.columns:
    airline_stats = (
        df.groupBy('airline')
        .agg(
            spark_round(avg('arr_delay'), 2).alias('avg_arr'),
            spark_round(avg('dep_delay'), 2).alias('avg_dep'),
            count('*').alias('flights'),
            spark_round(
                count(when(col('arr_delay') > 15, 1)) * 100.0 / count('*'), 1
            ).alias('pct_delayed')
        )
        .filter(col('flights') >= 10000)
        .orderBy('avg_arr', ascending=False)
        .toPandas()
    )

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle('Airline Performance Analysis — All Years', fontsize=15, fontweight='bold')

    c = [PAL[1] if v >= 0 else PAL[2] for v in airline_stats['avg_arr']]
    axes[0,0].barh(airline_stats['airline'], airline_stats['avg_arr'],
                   color=c, edgecolor='white')
    axes[0,0].axvline(0, color='black', lw=0.8)
    axes[0,0].set_xlabel('Avg Arrival Delay (min)')
    axes[0,0].set_title('Avg Arrival Delay per Airline')

    axes[0,1].barh(airline_stats['airline'], airline_stats['pct_delayed'],
                   color=PAL[3], edgecolor='white')
    axes[0,1].set_xlabel('% Flights Delayed >15 min')
    axes[0,1].set_title('Delay Rate per Airline')

    axes[1,0].scatter(airline_stats['flights']/1e6, airline_stats['avg_arr'],
                      s=120, color=PAL[0], alpha=0.8, edgecolors='white')
    for _, row in airline_stats.iterrows():
        axes[1,0].annotate(row['airline'],
                           (row['flights']/1e6, row['avg_arr']),
                           fontsize=8, alpha=0.8)
    axes[1,0].set_xlabel('Total Flights (millions)')
    axes[1,0].set_ylabel('Avg Arrival Delay (min)')
    axes[1,0].set_title('Volume vs Average Delay')

    axes[1,1].scatter(airline_stats['avg_dep'], airline_stats['avg_arr'],
                      s=120, color=PAL[4], alpha=0.8, edgecolors='white')
    for _, row in airline_stats.iterrows():
        axes[1,1].annotate(row['airline'],
                           (row['avg_dep'], row['avg_arr']),
                           fontsize=8, alpha=0.8)
    axes[1,1].set_xlabel('Avg Departure Delay (min)')
    axes[1,1].set_ylabel('Avg Arrival Delay (min)')
    axes[1,1].set_title('Dep Delay vs Arr Delay by Airline')

    plt.tight_layout()
    plt.savefig('viz_05_airlines.png', dpi=150, bbox_inches='tight')
    plt.show()

    print('\nFull airline table:')
    print(airline_stats.to_string(index=False))

In [ ]:
# ── VIZ 6: Day-of-week analysis ───────────────────────────
if 'day_of_week' in df.columns:
    dow_stats = (
        df.groupBy('day_of_week')
        .agg(spark_round(avg('arr_delay'), 2).alias('avg_arr'),
             count('*').alias('flights'))
        .orderBy('day_of_week')
        .toPandas().dropna()
    )
    day_names = {1:'Sun',2:'Mon',3:'Tue',4:'Wed',5:'Thu',6:'Fri',7:'Sat'}
    dow_stats['day'] = dow_stats['day_of_week'].map(day_names)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle('Day-of-Week Patterns', fontsize=15, fontweight='bold')

    worst = dow_stats['avg_arr'].idxmax()
    c_dow = [PAL[1] if i == worst else PAL[0] for i in dow_stats.index]
    axes[0].bar(dow_stats['day'], dow_stats['avg_arr'], color=c_dow, edgecolor='white')
    axes[0].set_ylabel('Avg Arrival Delay (min)')
    axes[0].set_title('Worst Day to Fly?')

    axes[1].bar(dow_stats['day'], dow_stats['flights']/1e6,
                color=PAL[2], edgecolor='white')
    axes[1].set_ylabel('Flights (millions)')
    axes[1].set_title('Busiest Day of Week')

    plt.tight_layout()
    plt.savefig('viz_06_day_of_week.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ── VIZ 7: Top routes ─────────────────────────────────────
if 'origin' in df.columns and 'dest' in df.columns:
    routes = (
        df.withColumn('route', concat_ws(' → ', col('origin'), col('dest')))
        .groupBy('route')
        .agg(spark_round(avg('arr_delay'), 1).alias('avg_delay'),
             count('*').alias('flights'))
        .filter(col('flights') >= 5000)
    )
    top_delayed  = routes.orderBy('avg_delay', ascending=False).limit(15).toPandas()
    most_flights = routes.orderBy('flights',   ascending=False).limit(15).toPandas()

    fig, axes = plt.subplots(1, 2, figsize=(17, 7))
    fig.suptitle('Route Analysis', fontsize=15, fontweight='bold')

    axes[0].barh(top_delayed['route'][::-1], top_delayed['avg_delay'][::-1],
                 color=PAL[1], edgecolor='white')
    axes[0].set_xlabel('Avg Arrival Delay (min)')
    axes[0].set_title('Top 15 Most Delayed Routes')

    axes[1].barh(most_flights['route'][::-1], most_flights['flights'][::-1]/1e3,
                 color=PAL[0], edgecolor='white')
    axes[1].set_xlabel('Flights (thousands)')
    axes[1].set_title('Top 15 Busiest Routes')

    plt.tight_layout()
    plt.savefig('viz_07_routes.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ── VIZ 8: Correlation heatmap ────────────────────────────
num_for_corr = [c for c in
    ['arr_delay','dep_delay','distance','air_time','taxi_out','dep_hour','month','day_of_week']
    if c in df.columns]

corr_pd = to_pd(df.select(num_for_corr), n=30000).dropna()
corr_matrix = corr_pd.corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    corr_matrix, annot=True, fmt='.2f',
    cmap='RdYlGn', center=0,
    square=True, ax=ax,
    linewidths=0.5, cbar_kws={'shrink':0.8},
    annot_kws={'size': 9}
)
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('viz_08_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n💡 Correlation with arr_delay:')
for c in [x for x in num_for_corr if x != 'arr_delay']:
    print(f'  {c:<18} {corr_matrix.loc["arr_delay",c]:+.3f}')

In [ ]:
# ── VIZ 9: dep_delay vs arr_delay scatter ─────────────────
scat = to_pd(df.select('dep_delay','arr_delay','distance'), n=25000)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Key Relationships', fontsize=15, fontweight='bold')

sc = axes[0].scatter(
    scat['dep_delay'], scat['arr_delay'],
    c=scat['distance'], cmap='viridis',
    alpha=0.25, s=6
)
plt.colorbar(sc, ax=axes[0], label='Distance (miles)')
axes[0].set_xlabel('Dep Delay (min)')
axes[0].set_ylabel('Arr Delay (min)')
axes[0].set_title('Dep Delay vs Arr Delay\n(coloured by distance)')

axes[1].hexbin(scat['distance'], scat['arr_delay'],
               gridsize=40, cmap='YlOrRd', mincnt=1)
axes[1].set_xlabel('Distance (miles)')
axes[1].set_ylabel('Arr Delay (min)')
axes[1].set_title('Distance vs Arr Delay (hexbin density)')

plt.tight_layout()
plt.savefig('viz_09_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── VIZ 10: Descriptive stats summary ─────────────────────
desc_cols = [c for c in ['arr_delay','dep_delay','distance','air_time'] if c in df.columns]
stats_rows = []
for c in desc_cols:
    row = df.select(c).agg(
        spark_round(avg(c),2).alias('mean'),
        spark_round(stddev(c),2).alias('std'),
        spark_round(spark_min(c),2).alias('min'),
        spark_round(percentile_approx(c,0.25),2).alias('q1'),
        spark_round(percentile_approx(c,0.50),2).alias('median'),
        spark_round(percentile_approx(c,0.75),2).alias('q3'),
        spark_round(spark_max(c),2).alias('max'),
    ).toPandas()
    row.insert(0,'column',c)
    stats_rows.append(row)

stats_df = pd.concat(stats_rows, ignore_index=True)
print('=== Descriptive Statistics ===')
print(stats_df.to_string(index=False))

---
## 🤖 Section 5 — Feature Engineering & ML Preparation

In [ ]:
from pyspark.ml.feature import VectorAssembler, StringIndexer, StandardScaler
from pyspark.ml import Pipeline

stages       = []
cat_features = []

# Encode airline, origin, dest → numeric index
for cat_col in ['airline', 'origin', 'dest']:
    if cat_col in df.columns:
        idx = StringIndexer(
            inputCol=cat_col,
            outputCol=f'{cat_col}_idx',
            handleInvalid='keep'
        )
        stages.append(idx)
        cat_features.append(f'{cat_col}_idx')

# Build full feature list
num_features = ['dep_delay', 'distance']
num_features += [c for c in ['air_time','taxi_out','dep_hour','month','day_of_week']
                 if c in df.columns]

FEATURE_COLS = num_features + cat_features
LABEL_COL    = 'arr_delay'

assembler = VectorAssembler(
    inputCols=FEATURE_COLS,
    outputCol='features',
    handleInvalid='skip'
)
stages.append(assembler)

print(f'Feature columns ({len(FEATURE_COLS)}): {FEATURE_COLS}')
print(f'Label: {LABEL_COL}')

In [ ]:
# Build prep pipeline
prep_model = Pipeline(stages=stages).fit(df)
df_ml      = prep_model.transform(df).select('features', LABEL_COL)

# 80/20 split
train_df, test_df = df_ml.randomSplit([0.8, 0.2], seed=42)
train_df.cache()
test_df.cache()

print(f'Train rows : {train_df.count():,}')
print(f'Test rows  : {test_df.count():,}')
df_ml.show(3, truncate=60)

---
## 🧠 Section 6 — Training 4 ML Models

| # | Model | Method | Key property |
|---|-------|--------|--------------|
| 1 | **Linear Regression** | `.fit()` — Estimator | Fast, interpretable baseline |
| 2 | **Decision Tree** | `.fit()` — Estimator | Rule-based, easy to explain |
| 3 | **Random Forest** | `.fit()` — Estimator | Average of 50 trees, robust |
| 4 | **Gradient Boosted Trees** | `.fit()` — Estimator | Sequential correction, usually best |

In [ ]:
from pyspark.ml.regression import (
    LinearRegression, DecisionTreeRegressor,
    RandomForestRegressor, GBTRegressor
)
from pyspark.ml.evaluation import RegressionEvaluator

def evaluate(preds):
    metrics = {}
    for metric in ['rmse','mae','r2']:
        ev = RegressionEvaluator(
            labelCol=LABEL_COL, predictionCol='prediction', metricName=metric)
        metrics[metric] = ev.evaluate(preds)
    return metrics

results = {}
print('Ready to train ✅')

# Model 1: Linear Regression 

In [ ]:
t0 = time.time()
lr_model = LinearRegression(
    featuresCol='features', labelCol=LABEL_COL,
    predictionCol='prediction',
    maxIter=100, regParam=0.1
).fit(train_df)
lr_preds = lr_model.transform(test_df)
results['Linear Regression'] = {**evaluate(lr_preds),
    'time': round(time.time()-t0,1), 'preds': lr_preds, 'model': lr_model}
r = results['Linear Regression']
print(f'✅ RMSE={r["rmse"]:.3f}  MAE={r["mae"]:.3f}  R²={r["r2"]:.4f}  ({r["time"]}s)')

In [ ]:
# ── Model 2: Decision Tree ─────────────────────────────────
t0 = time.time()
dt_model = DecisionTreeRegressor(
    featuresCol='features', labelCol=LABEL_COL,
    predictionCol='prediction',
    maxDepth=10, minInstancesPerNode=20
).fit(train_df)
dt_preds = dt_model.transform(test_df)
results['Decision Tree'] = {**evaluate(dt_preds),
    'time': round(time.time()-t0,1), 'preds': dt_preds, 'model': dt_model,
    'importances': dt_model.featureImportances}
r = results['Decision Tree']
print(f'✅ RMSE={r["rmse"]:.3f}  MAE={r["mae"]:.3f}  R²={r["r2"]:.4f}  ({r["time"]}s)')

In [ ]:
# ── Model 3: Random Forest ─────────────────────────────────
print('⏳ Training Random Forest (may take 2–4 min on full data)...')
t0 = time.time()
rf_model = RandomForestRegressor(
    featuresCol='features', labelCol=LABEL_COL,
    predictionCol='prediction',
    numTrees=50, maxDepth=10, seed=42,
    subsamplingRate=0.8
).fit(train_df)
rf_preds = rf_model.transform(test_df)
results['Random Forest'] = {**evaluate(rf_preds),
    'time': round(time.time()-t0,1), 'preds': rf_preds, 'model': rf_model,
    'importances': rf_model.featureImportances}
r = results['Random Forest']
print(f'✅ RMSE={r["rmse"]:.3f}  MAE={r["mae"]:.3f}  R²={r["r2"]:.4f}  ({r["time"]}s)')

In [ ]:
# ── Model 4: Gradient Boosted Trees ───────────────────────
print('⏳ Training GBT (may take 3–6 min on full data)...')
t0 = time.time()
gbt_model = GBTRegressor(
    featuresCol='features', labelCol=LABEL_COL,
    predictionCol='prediction',
    maxIter=50, maxDepth=8,
    stepSize=0.1, subsamplingRate=0.8, seed=42
).fit(train_df)
gbt_preds = gbt_model.transform(test_df)
results['GBT'] = {**evaluate(gbt_preds),
    'time': round(time.time()-t0,1), 'preds': gbt_preds, 'model': gbt_model,
    'importances': gbt_model.featureImportances}
r = results['GBT']
print(f'✅ RMSE={r["rmse"]:.3f}  MAE={r["mae"]:.3f}  R²={r["r2"]:.4f}  ({r["time"]}s)')

---
## 7 — Model Comparison & Evaluation

In [ ]:
# ── Summary table ──────────────────────────────────────────
summary = pd.DataFrame([
    {'Model': n, 'RMSE': round(m['rmse'],3),
     'MAE': round(m['mae'],3), 'R²': round(m['r2'],4),
     'Train Time (s)': m['time']}
    for n, m in results.items()
]).sort_values('RMSE').reset_index(drop=True)
summary.insert(0, 'Rank', range(1, len(summary)+1))
BEST = summary.iloc[0]['Model']

print('═'*65)
print('        MODEL COMPARISON — Lower RMSE/MAE = Better')
print('═'*65)
print(summary.to_string(index=False))
print('═'*65)
print(f'\n Best model: {BEST}')

In [ ]:
# ── VIZ: 3-metric comparison bars ─────────────────────────
models      = summary['Model'].tolist()
bar_colors  = [PAL[2] if m == BEST else PAL[0] for m in models]

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('Model Comparison — All 4 Algorithms', fontsize=15, fontweight='bold')

for ax, metric, title in zip(
    axes,
    ['RMSE','MAE','R²'],
    ['RMSE  (lower = better)','MAE  (lower = better)','R²  (higher = better)']
):
    vals  = summary[metric].tolist()
    bars  = ax.bar(models, vals, color=bar_colors, edgecolor='white', width=0.5)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2,
                bar.get_height() + max(vals)*0.012,
                f'{v:.3f}', ha='center', fontsize=10, fontweight='bold')
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=12)
    # Gold border on winner
    bars[models.index(BEST)].set_edgecolor('#FFC107')
    bars[models.index(BEST)].set_linewidth(2.5)

plt.tight_layout()
plt.savefig('viz_10_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── VIZ: Actual vs Predicted + Residuals (best model) ─────
best_preds = results[BEST]['preds']
sp = (
    best_preds.select(LABEL_COL, 'prediction')
    .sample(fraction=min(1.0, 15000/test_df.count()), seed=7)
    .toPandas().dropna()
)
sp['residual'] = sp[LABEL_COL] - sp['prediction']

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle(f'Best Model — {BEST}: Detailed Analysis',
             fontsize=14, fontweight='bold')

lim = max(abs(sp[LABEL_COL].quantile(0.99)),
          abs(sp['prediction'].quantile(0.99)))
axes[0].scatter(sp[LABEL_COL], sp['prediction'],
                alpha=0.25, s=8, color=PAL[2])
axes[0].plot([-lim,lim],[-lim,lim],'r--',lw=1.5,label='Perfect')
axes[0].set_xlim(-lim,lim); axes[0].set_ylim(-lim,lim)
axes[0].set_xlabel('Actual Delay (min)')
axes[0].set_ylabel('Predicted Delay (min)')
axes[0].set_title('Actual vs Predicted')
axes[0].legend()

axes[1].hist(sp['residual'], bins=70, color=PAL[3],
             edgecolor='white', linewidth=0.4)
axes[1].axvline(0, color='red', lw=1.5, linestyle='--', label='Zero error')
axes[1].axvline(sp['residual'].mean(), color='gold', lw=1.5,
                linestyle='-.', label=f'Mean residual: {sp["residual"].mean():.2f}')
axes[1].set_xlabel('Residual (Actual − Predicted)')
axes[1].set_ylabel('Count')
axes[1].set_title('Residual Distribution')
axes[1].legend()

plt.tight_layout()
plt.savefig('viz_11_best_model.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── VIZ: Feature importances (all tree models) ────────────
tree_results = {k: v for k, v in results.items() if 'importances' in v}
n = len(tree_results)

if n > 0:
    fig, axes = plt.subplots(1, n, figsize=(8*n, 5))
    if n == 1: axes = [axes]
    fig.suptitle('Feature Importances — Tree Models', fontsize=14, fontweight='bold')

    for ax, (name, m) in zip(axes, tree_results.items()):
        imp = pd.DataFrame({
            'feature':    FEATURE_COLS,
            'importance': m['importances'].toArray()
        }).sort_values('importance')

        c_imp = [PAL[0] if v < imp['importance'].median() else PAL[2]
                 for v in imp['importance']]
        ax.barh(imp['feature'], imp['importance'], color=c_imp, edgecolor='white')
        for i, (feat, val) in enumerate(zip(imp['feature'], imp['importance'])):
            ax.text(val+0.001, i, f'{val:.3f}', va='center', fontsize=9)
        ax.set_xlabel('Importance')
        ax.set_title(name)

    plt.tight_layout()
    plt.savefig('viz_12_feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()

---
## 8 — Conclusion

In [ ]:
best_rmse = results[BEST]['rmse']
best_r2   = results[BEST]['r2']

print('═'*65)
print('        US FLIGHT DELAY PREDICTOR — FINAL SUMMARY')
print('═'*65)
print(f'  Dataset          : US Domestic Flights — Full Range')
print(f'  Total rows used  : {TOTAL_ROWS:,}')
print(f'  Features         : {FEATURE_COLS}')
print(f'  Target           : {LABEL_COL}')
print()
print('  All model results (ranked):')
for _, row in summary.iterrows():
    marker = '  ← WINNER 🏆' if row['Model'] == BEST else ''
    print(f"  {int(row['Rank'])}. {row['Model']:<22}"
          f"  RMSE={row['RMSE']:.2f}  MAE={row['MAE']:.2f}  R²={row['R²']:.3f}{marker}")
print()
print(f'  Winner  : {BEST}')
print(f'  RMSE    : {best_rmse:.2f} min  → on average predictions are off by ~{best_rmse:.0f} min')
print(f'  R²      : {best_r2:.3f}  → model explains {best_r2*100:.1f}% of delay variance')
print('═'*65)

# Release caches
train_df.unpersist()
test_df.unpersist()
df.unpersist()
print('\n✅ All Spark caches released')